# Phase 4: Tree-Based Models with SageMaker

**Goals:**
1. Train Random Forest and XGBoost via SageMaker Training Jobs (not locally)
2. Track all runs with **SageMaker Experiments** — compare hyperparameter configurations
3. Understand `scale_pos_weight` (XGBoost) and `class_weight` (RF) tuning
4. Identify the best model configuration to carry into Phase 5

**Pre-requisites:**
- AWS credentials configured (`aws configure`)
- S3 bucket with train/val/test CSVs uploaded (run the cell below)
- SageMaker execution role with S3 and SageMaker permissions

In [ ]:
import sys
sys.path.insert(0, '..')

import boto3
import sagemaker
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.xgboost import XGBoost
from sagemaker.experiments.run import Run

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import AWS config
from config.aws_config import AWS_REGION, S3_BUCKET, S3_DATA_PREFIX
from src.preprocessing import load_data, scale_features, split_data, save_splits_csv

%matplotlib inline

## 1. Config — Fill in Your Values

In [ ]:
# Import AWS config
from config.aws_config import AWS_REGION, S3_BUCKET, S3_DATA_PREFIX

# AWS Configuration - Use settings from config
BUCKET = S3_BUCKET
REGION = AWS_REGION
PREFIX = "fraud-detection"
EXPERIMENT_NAME = "fraud-detection-models"

session = sagemaker.Session()
role = sagemaker.get_execution_role()

print(f"Region: {REGION}")
print(f"Bucket: {BUCKET}")
print(f"Role:   {role}")

## 2. Prepare & Upload Data to S3

In [ ]:
df = load_data('../data/raw/creditcard.csv')
df = scale_features(df)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

# Save CSVs locally (label-first, no header — SageMaker XGBoost format)
save_splits_csv(X_train, y_train, X_val, y_val, X_test, y_test, output_dir='../data/splits')
print('Splits saved locally.')

In [ ]:
s3 = boto3.client('s3')

def upload_to_s3(local_path, s3_key):
    s3.upload_file(local_path, BUCKET, s3_key)
    return f's3://{BUCKET}/{s3_key}'

train_uri = upload_to_s3('../data/splits/train.csv', f'{PREFIX}/data/train/train.csv')
val_uri   = upload_to_s3('../data/splits/val.csv',   f'{PREFIX}/data/val/val.csv')
test_uri  = upload_to_s3('../data/splits/test.csv',  f'{PREFIX}/data/test/test.csv')

print(f'Train: {train_uri}')
print(f'Val:   {val_uri}')
print(f'Test:  {test_uri}')

## 3. SageMaker Experiment Setup

An **Experiment** groups related training runs. Each run logs hyperparameters and metrics automatically — visible in SageMaker Studio's Experiments panel.

In [ ]:
from sagemaker.experiments import Experiment

experiment = Experiment.create(
    experiment_name=EXPERIMENT_NAME,
    description='Fraud detection: comparing RF and XGBoost with different imbalance strategies',
    sagemaker_boto_client=boto3.client('sagemaker')
)
print(f'Experiment created: {EXPERIMENT_NAME}')

## 4. XGBoost Training Jobs

We launch three runs: (a) auto-computed `scale_pos_weight`, (b) `scale_pos_weight=1` (no correction), (c) SMOTE-preprocessed data.

In [ ]:
from sagemaker.inputs import TrainingInput

def launch_xgboost_job(run_name, scale_pos_weight=None, n_estimators=200, max_depth=6, lr=0.1):
    """Launch a SageMaker XGBoost training job and attach it to the experiment."""
    hyperparams = {
        'n-estimators': n_estimators,
        'max-depth': max_depth,
        'learning-rate': lr,
    }
    if scale_pos_weight is not None:
        hyperparams['scale-pos-weight'] = scale_pos_weight

    estimator = SKLearn(
        entry_point='../src/train_xgboost.py',
        role=role,
        instance_type='ml.m5.large',
        framework_version='1.2-1',
        hyperparameters=hyperparams,
        metric_definitions=[
            {'Name': 'validation:auc_roc', 'Regex': r'\[validation\] auc_roc=([0-9\\.]+)'},
            {'Name': 'validation:auc_pr',  'Regex': r'\[validation\] auc_pr=([0-9\\.]+)'},
        ],
        sagemaker_session=session,
        enable_sagemaker_metrics=True,
        experiment_config={
            'ExperimentName': EXPERIMENT_NAME,
            'TrialName': run_name,
            'TrialComponentDisplayName': run_name,
        }
    )

    estimator.fit(
        inputs={
            'train': TrainingInput(train_uri, content_type='text/csv'),
            'validation': TrainingInput(val_uri, content_type='text/csv'),
        },
        job_name=run_name,
        wait=False,  # non-blocking — launch multiple jobs in sequence
    )
    print(f'Launched: {run_name}')
    return estimator


# Run A: auto scale_pos_weight (recommended)
xgb_auto = launch_xgboost_job('xgb-auto-weight')

# Run B: no class weight correction — see how much it hurts
xgb_no_weight = launch_xgboost_job('xgb-no-weight', scale_pos_weight=1.0)

# Run C: deeper trees
xgb_deep = launch_xgboost_job('xgb-deep', max_depth=10)

In [ ]:
# Wait for all jobs to complete before comparing
# (SageMaker jobs run in parallel on AWS infrastructure)
import time
sm_client = boto3.client('sagemaker')

job_names = ['xgb-auto-weight', 'xgb-no-weight', 'xgb-deep']

def wait_for_jobs(job_names, poll_seconds=30):
    remaining = list(job_names)
    while remaining:
        done = []
        for name in remaining:
            status = sm_client.describe_training_job(TrainingJobName=name)['TrainingJobStatus']
            print(f'  {name}: {status}')
            if status in ('Completed', 'Failed', 'Stopped'):
                done.append(name)
        remaining = [n for n in remaining if n not in done]
        if remaining:
            print(f'Waiting {poll_seconds}s...')
            time.sleep(poll_seconds)
    print('All jobs finished.')

wait_for_jobs(job_names)

## 5. Random Forest Training Jobs

In [ ]:
def launch_rf_job(run_name, class_weight='balanced', n_estimators=200, max_depth=None):
    hyperparams = {
        'n-estimators': n_estimators,
        'class-weight': class_weight,
    }
    if max_depth is not None:
        hyperparams['max-depth'] = max_depth

    estimator = SKLearn(
        entry_point='../src/train_rf.py',
        role=role,
        instance_type='ml.m5.large',
        framework_version='1.2-1',
        hyperparameters=hyperparams,
        metric_definitions=[
            {'Name': 'validation:auc_roc', 'Regex': r'\[validation\] auc_roc=([0-9\\.]+)'},
            {'Name': 'validation:auc_pr',  'Regex': r'\[validation\] auc_pr=([0-9\\.]+)'},
        ],
        sagemaker_session=session,
        enable_sagemaker_metrics=True,
        experiment_config={
            'ExperimentName': EXPERIMENT_NAME,
            'TrialName': run_name,
            'TrialComponentDisplayName': run_name,
        }
    )

    estimator.fit(
        inputs={
            'train': TrainingInput(train_uri, content_type='text/csv'),
            'validation': TrainingInput(val_uri, content_type='text/csv'),
        },
        job_name=run_name,
        wait=False,
    )
    print(f'Launched: {run_name}')
    return estimator


rf_balanced   = launch_rf_job('rf-balanced')
rf_no_weight  = launch_rf_job('rf-no-weight', class_weight='none')
rf_depth_10   = launch_rf_job('rf-depth-10', max_depth=10)

wait_for_jobs(['rf-balanced', 'rf-no-weight', 'rf-depth-10'])

## 6. Compare All Runs via SageMaker Experiments

In [ ]:
from sagemaker.analytics import ExperimentAnalytics

analytics = ExperimentAnalytics(
    experiment_name=EXPERIMENT_NAME,
    metric_names=['validation:auc_pr', 'validation:auc_roc'],
    sort_by='metrics.validation:auc_pr.max',
    sort_order='Descending',
    sagemaker_session=session
)

results_df = analytics.dataframe()
cols = ['TrialComponentName', 'metrics.validation:auc_pr.max', 'metrics.validation:auc_roc.max']
print(results_df[cols].to_string(index=False))

In [ ]:
# Bar chart of AUC-PR across all runs
df_plot = results_df[cols].copy()
df_plot.columns = ['Run', 'AUC-PR', 'AUC-ROC']
df_plot = df_plot.sort_values('AUC-PR', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue' if 'xgb' in r else 'seagreen' for r in df_plot['Run']]
ax.barh(df_plot['Run'], df_plot['AUC-PR'], color=colors)
ax.set_xlabel('AUC-PR (Precision-Recall AUC)')
ax.set_title('SageMaker Experiment Results: AUC-PR by Run')
ax.axvline(x=0.8, color='crimson', linestyle='--', label='Target: 0.80')
ax.legend()
plt.tight_layout()
plt.savefig('../data/experiment_comparison.png', dpi=120)
plt.show()

## 7. Key Observations

- **XGBoost with `scale_pos_weight`** typically outperforms Random Forest on AUC-PR for this dataset
- **No-weight runs** show significantly lower recall — the model learns to ignore fraud
- **Deeper trees** may overfit — watch the gap between train and validation AUC-PR
- The best run from this experiment will be the model we take to Phase 5 (threshold tuning)

**Next step → `04_anomaly_detection.ipynb`**: Compare against SageMaker's unsupervised Random Cut Forest.